# **Prototype networks: «this looks like that»**### Practice for the ProtoPNet lesson, block «Non-linear interpretable models»Here we build a ProtoPNet end to end and by hand — on a task that runs on a CPU in a couple ofminutes: three Fashion-MNIST classes and two prototypes per class.What sets it apart from the other methods of the course is that the explanation is not builtafter the answer and is not derived from the structure. It **is the computation itself**: thenetwork assembles the prediction out of similarities to exemplars, and those similarities canbe shown to the reader.Step by step we train the backbone together with the prototypes, run **push** — the operationeverything was set up for — and see what it does to the accuracy. Then we tune a single lastlayer and read the explanation of a particular image as «similarity x weight».Enjoy the work!

In [ ]:
import numpy as npimport torchimport torch.nn as nnimport torch.nn.functional as Fimport matplotlib.pyplot as pltfrom torchvision import datasets, transformstorch.manual_seed(0)np.random.seed(0)CLASSES = [0, 1, 9]          # t-shirt, trousers, ankle bootPROTO_PER_CLASS = 2          # prototypes per classD = 64                       # length of a cell vector and of a prototypeEPS = 1e-4

## The dataWe take three dissimilar classes so that the prototypes are distinguishable by eye: a t-shirt,trousers and an ankle boot. 600 images per class for training and 150 for testing — enough tosee every effect and little enough not to wait.

In [ ]:
tf = transforms.Compose([transforms.Resize(64), transforms.ToTensor()])train_full = datasets.FashionMNIST('./data', train=True, download=True, transform=tf)test_full = datasets.FashionMNIST('./data', train=False, download=True, transform=tf)def subset(ds, n_per_class):    idx, seen = [], {c: 0 for c in CLASSES}    for i, y in enumerate(ds.targets.tolist()):        if y in seen and seen[y] < n_per_class:            idx.append(i); seen[y] += 1        if all(v >= n_per_class for v in seen.values()):            break    return torch.utils.data.Subset(ds, idx)train_ds, test_ds = subset(train_full, 600), subset(test_full, 150)remap = {c: i for i, c in enumerate(CLASSES)}loader = lambda ds, bs, sh: torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=sh)print(len(train_ds), len(test_ds))

## The modelThree parts, exactly as in the lesson.**The backbone $f$** is a convolutional network without a classification head. Its output is a$D \times H \times W$ tensor: each of the $H \cdot W$ cells describes its own patch of theimage by a vector of length $D$.**The prototype layer** holds $m$ learnable vectors of the same length $D$, two per class. Eachis attached to its own class.**The linear layer $h$** takes $m$ similarities and outputs class scores. The weights do notstart at random: $1$ for the prototypes of the layer's own class and $-0.5$ for the others —that immediately sets the intended reading of the explanation.The similarity is computed by the formula from the lesson:$$g_{p_j}(z) = \max_{\tilde z} \log\left(\frac{\|\tilde z - p_j\|^2 + 1}{\|\tilde z - p_j\|^2 + \varepsilon}\right)$$The logarithm decreases with distance, and the maximum over cells answers the question «isthere anywhere on the image a patch similar to this exemplar» — and remembers where.

In [ ]:
class ProtoPNet(nn.Module):    def __init__(self, n_classes, per_class):        super().__init__()        self.n_classes, self.per_class = n_classes, per_class        self.m = n_classes * per_class        # backbone f: outputs a D x H x W map, every cell describes one patch of the image        self.features = nn.Sequential(            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),            nn.Conv2d(64, D, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),            nn.Conv2d(D, D, 1), nn.Sigmoid(),        )        # prototype layer: m learnable vectors, per_class of them for every class        self.prototypes = nn.Parameter(torch.rand(self.m, D))        self.identity = torch.zeros(self.m, n_classes)        for j in range(self.m):            self.identity[j, j // per_class] = 1        # linear layer h: 1 for own-class prototypes, -0.5 for the others        self.last = nn.Linear(self.m, n_classes, bias=False)        with torch.no_grad():            self.last.weight.copy_((self.identity * 1.5 - 0.5).t())    def similarity(self, z):        B, _, H, W = z.shape        patches = z.permute(0, 2, 3, 1).reshape(B, H * W, D)        d2 = torch.cdist(patches, self.prototypes.unsqueeze(0).expand(B, -1, -1)) ** 2        sim = # Your code here: the similarity formula from the lesson, log((d2 + 1) / (d2 + EPS))        return sim, d2    def forward(self, x):        z = self.features(x)        sim, d2 = self.similarity(z)        pooled = sim.max(dim=1).values      # max over cells: was a similar patch found anywhere        return self.last(pooled), pooled, d2, zmodel = ProtoPNet(len(CLASSES), PROTO_PER_CLASS)sum(p.numel() for p in model.parameters())

## Stage 1: the backbone and the prototypesTwo terms from Chen et al. are added to the cross-entropy:- **the clustering term** — every image must have a cell close to at least one prototype of its  own class;- **the separation term** — cells must be kept away from the prototypes of other classes.The first gathers the prototypes into meaningful exemplars, the second stops them from becomingcommon to every class.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)for epoch in range(3):    model.train(); tot = correct = 0    for x, y in loader(train_ds, 64, True):        y = torch.tensor([remap[int(v)] for v in y])        logits, pooled, d2, _ = model(x)        ce = F.cross_entropy(logits, y)        ident = model.identity[:, y].t()          # 1 for the prototypes of the true class        min_d = d2.min(dim=1).values              # nearest cell for every prototype        clst = (min_d * ident).sum(1).mean()      # clustering term: own prototypes must be closer        sep = # Your code here: the separation term, keep other prototypes further away        loss = ce + 0.8 * clst - 0.08 * sep        opt.zero_grad(); loss.backward(); opt.step()        tot += len(y); correct += (logits.argmax(1) == y).sum().item()    print(f'epoch {epoch + 1}: accuracy {correct / tot:.3f}')

In [ ]:
@torch.no_grad()def accuracy(ds):    model.eval(); ok = n = 0    for x, y in loader(ds, 128, False):        y = torch.tensor([remap[int(v)] for v in y])        ok += (model(x)[0].argmax(1) == y).sum().item(); n += len(y)    return ok / nacc_before = accuracy(test_ds)print(f'before push: {acc_before:.3f}')

## Push: a prototype becomes a piece of a real imageWhile a prototype is just a learnable vector, there is nothing to show the reader: «similarity0.91 with the vector $p_1$» is not an explanation. Push replaces every prototype with **thenearest cell among the training images of its class**.After that the prototype stops being an abstraction: it is a specific patch of a specific imagethat can be cut out and printed next to the explanation.The price is known in advance — watch the accuracy before and after.

In [ ]:
@torch.no_grad()def push():    """Every prototype is replaced by the nearest cell of a training image of its class."""    model.eval()    best = {j: (float('inf'), None, None) for j in range(model.m)}    for x, y in loader(train_ds, 64, False):        y = torch.tensor([remap[int(v)] for v in y])        z = model.features(x)        B, _, H, W = z.shape        patches = z.permute(0, 2, 3, 1).reshape(B, H * W, D)        for j in range(model.m):            mask = (y == j // model.per_class)     # images of its own class only            if not mask.any():                continue            p = patches[mask]            d = ((p - model.prototypes[j]) ** 2).sum(-1)            v, flat = d.view(-1).min(0)            if v.item() < best[j][0]:                ni, cell = divmod(int(flat), H * W)                best[j] = (v.item(), p[ni, cell].clone(), (x[mask][ni], cell, H, W))    for j, (_, vec, _) in best.items():        model.prototypes.data[j] = vec            # the prototype is now a piece of a real image    return bestsources = push()acc_after = accuracy(test_ds)print(f'after push: {acc_after:.3f}   (change {acc_after - acc_before:+.3f})')

## Stage 3: the last layer onlyThe backbone and the prototypes are frozen, a single $h$ is tuned — the problem is convex. The$L_1$ penalty pushes the weights of **other classes'** prototypes towards zero, so only thearguments «for» remain in the explanation, without the bookkeeping of negative contributions.

In [ ]:
for p in model.features.parameters():    p.requires_grad = Falsemodel.prototypes.requires_grad = False            # backbone and prototypes are frozenopt2 = torch.optim.Adam(model.last.parameters(), lr=1e-3)for epoch in range(2):    model.train()    for x, y in loader(train_ds, 64, True):        y = torch.tensor([remap[int(v)] for v in y])        logits, *_ = model(x)        l1 = # Your code here: L1 penalty on the weights of OTHER prototypes, (1 - model.identity.t())        loss = F.cross_entropy(logits, y) + 1e-3 * l1        opt2.zero_grad(); loss.backward(); opt2.step()print(f'after fine-tuning: {accuracy(test_ds):.3f}')

## Reading the explanationThe contribution of a prototype to the prediction is its **similarity multiplied by the weight**in the last layer. Prototypes of the predicted class give a plus, the others a minus: that isexactly how the layer was initialised.

In [ ]:
model.eval()x, y = test_ds[0]with torch.no_grad():    logits, pooled, _, _ = model(x.unsqueeze(0))pred = int(logits.argmax())contrib = (pooled[0] * model.last.weight[pred]).detach().numpy()order = np.argsort(-contrib)print(f'true class: {CLASSES[remap[int(y)]]}, prediction: {CLASSES[pred]}\n')for j in order[:4]:    own = 'own' if j // PROTO_PER_CLASS == pred else 'other'    print(f'prototype {j} ({own} class): similarity {pooled[0][j]:6.3f} '          f'x weight {model.last.weight[pred][j]:6.3f} = {contrib[j]:6.3f}')

## Showing what it looks likeThree pictures side by side: the image itself, the similarity map for the best prototype (whereexactly the network found a similar patch), and the patch of the training image this prototypecame from during push.This is the explanation of the form «this part of your image looks like this exemplar».

In [ ]:
with torch.no_grad():    z = model.features(x.unsqueeze(0))    sim, _ = model.similarity(z)H = W = z.shape[-1]j = int(order[0])smap = sim[0, :, j].reshape(H, W).numpy()src_img, src_cell, sH, sW = sources[j][2]r, c = divmod(src_cell, sW)step = src_img.shape[-1] // sHfig, ax = plt.subplots(1, 3, figsize=(11, 3.6))ax[0].imshow(x.squeeze(), cmap='gray'); ax[0].set_title('image being explained')ax[1].imshow(x.squeeze(), cmap='gray')ax[1].imshow(np.kron(smap, np.ones((step, step))), alpha=0.5, cmap='jet')ax[1].set_title(f'similarity to prototype {j}')ax[2].imshow(src_img.squeeze()[r * step:(r + 1) * step, c * step:(c + 1) * step], cmap='gray')ax[2].set_title('source patch of the prototype')for a in ax:    a.axis('off')plt.tight_layout(); plt.show()

## Tasks1. **The drop after push.** Compare the accuracy before and after push and explain the sign of   the difference. Why does a prototype that has become a piece of a real image make the   prediction worse — and why is that not considered a failure?2. **The price of interpretability.** Train the same backbone with an ordinary linear head   instead of the prototype layer (replace `ProtoPNet` with   `nn.Sequential(features, Flatten, Linear)`). How much does the accuracy differ? That is the   price of being interpretable by construction.3. **The number of prototypes.** Set `PROTO_PER_CLASS = 1` and then `= 4`. How do the accuracy   and the readability of the explanation change? Where is the trade-off, in your view?4. **Prototypes of other classes.** Look at `model.last.weight` after stage 3. How many weights   of other classes' prototypes went to zero? What does that give the explanation?5. **The limits.** Take an image the model gets wrong and build the same explanation for it.   Which prototypes does it lean on? Does the explanation show why it erred?6. **A check under perturbation.** Shift the image by two or three pixels (`torch.roll`) and   recompute the similarities. How much did they change? This is exactly the check proposed by   Hoffmann et al. (2021) in «This Looks Like That… Does it?».

## What to take awayA prototype explanation is more honest than many: it is not bolted on from the outside butcoincides with the computation. Yet it has a limit of its own, and it lies where its strength is.**The similarity is computed in the latent space of the network, and it need not agree with thehuman one.** The frame on the similarity map shows where the maximum is, but it does not say whatexactly the prototype caught on — colour, shape or texture. It is a hint, not a measurement.Checking such explanations is proposed to be done with perturbations — task 6 is exactly aboutthat.